In [1]:
app_code = '''import streamlit as st
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px

st.set_page_config(
    page_title="SocialPulse - Brand Intelligence",
    page_icon=":bar_chart:",
    layout="wide"
)

st.markdown("""
    <style>
    .stApp { background-color: #0e1117; color: white; }
    .main-title { font-size: 42px; font-weight: bold; color: #00d4ff; }
    .subtitle { font-size: 16px; color: #888; }
    </style>
""", unsafe_allow_html=True)

st.markdown('<p class="main-title">SocialPulse - Brand Intelligence Platform</p>', unsafe_allow_html=True)
st.markdown('<p class="subtitle">Real-time sentiment, clustering and crisis detection for Indian startups</p>', unsafe_allow_html=True)

st.sidebar.title("SocialPulse")
page = st.sidebar.radio("Navigation", [
    "Overview", 
    "Sentiment Analysis", 
    "Brand Clusters", 
    "Crisis Detection"
])

st.write(f"Current page: {page}")
'''

with open("app.py", "w", encoding="utf-8") as f:
    f.write(app_code)

print("app.py saved successfully!")

app.py saved successfully!


In [2]:
overview_code = '''
# Load data once
@st.cache_data
def load_data():
    df = pd.read_csv("cleaned_reviews.csv")
    brand_features = pd.read_csv("brand_clusters.csv")
    return df, brand_features

df, brand_features = load_data()

if page == "Overview":
    st.subheader("Platform Overview")
    
    col1, col2, col3, col4 = st.columns(4)
    col1.metric("Total Reviews", f"{len(df):,}")
    col2.metric("Brands Tracked", df['brand'].nunique())
    col3.metric("Positive %", f"{(df['sentiment']=='Positive').mean()*100:.1f}%")
    col4.metric("Negative %", f"{(df['sentiment']=='Negative').mean()*100:.1f}%")
    
    st.markdown("---")
    
    col5, col6 = st.columns(2)
    with col5:
        sentiment_counts = df['sentiment'].value_counts()
        fig = px.pie(values=sentiment_counts.values, names=sentiment_counts.index,
                     color=sentiment_counts.index,
                     color_discrete_map={'Positive':'#00cc44','Negative':'#ff4444','Neutral':'#ffaa00'},
                     title="Overall Sentiment Distribution")
        fig.update_layout(paper_bgcolor="#0e1117", plot_bgcolor="#0e1117", font_color="white")
        st.plotly_chart(fig, use_container_width=True)
    
    with col6:
        avg_rating = df.groupby('brand')['rating'].mean().sort_values(ascending=False)
        fig2 = px.bar(x=avg_rating.values, y=avg_rating.index, orientation='h',
                      title="Average Rating by Brand",
                      labels={'x':'Average Rating','y':'Brand'})
        fig2.update_layout(paper_bgcolor="#0e1117", plot_bgcolor="#0e1117", font_color="white")
        st.plotly_chart(fig2, use_container_width=True)
'''

with open("app.py", "a", encoding="utf-8") as f:
    f.write(overview_code)

print("Overview page added!")

Overview page added!


In [3]:
sentiment_page_code = '''
if page == "Sentiment Analysis":
    st.subheader("Sentiment Analysis")
    
    st.markdown("### Try It Yourself")
    user_review = st.text_area("Enter a review to analyze:", 
                                 placeholder="Type something like 'delivery was late and food was cold'")
    
    if st.button("Analyze Sentiment"):
        if user_review.strip():
            model = pickle.load(open('sentiment_model.pkl', 'rb'))
            tfidf = pickle.load(open('tfidf_vectorizer.pkl', 'rb'))
            cleaned = user_review.lower()
            vectorized = tfidf.transform([cleaned])
            prediction = model.predict(vectorized)[0]
            proba = model.predict_proba(vectorized)[0]
            
            if prediction == "Positive":
                st.success(f"Predicted Sentiment: Positive (confidence: {max(proba)*100:.1f}%)")
            else:
                st.error(f"Predicted Sentiment: Negative (confidence: {max(proba)*100:.1f}%)")
        else:
            st.warning("Please enter a review first")
    
    st.markdown("---")
    st.markdown("### Sentiment Trends by Brand")
    
    selected_brand = st.selectbox("Select a brand", sorted(df['brand'].unique()))
    brand_df = df[df['brand'] == selected_brand]
    
    col1, col2 = st.columns(2)
    with col1:
        sent_counts = brand_df['sentiment'].value_counts()
        fig = px.bar(x=sent_counts.index, y=sent_counts.values,
                     color=sent_counts.index,
                     color_discrete_map={'Positive':'#00cc44','Negative':'#ff4444','Neutral':'#ffaa00'},
                     title=f"{selected_brand} - Sentiment Breakdown")
        fig.update_layout(paper_bgcolor="#0e1117", plot_bgcolor="#0e1117", font_color="white")
        st.plotly_chart(fig, use_container_width=True)
    
    with col2:
        st.metric("Average Rating", f"{brand_df['rating'].mean():.2f} / 5")
        st.metric("Total Reviews", f"{len(brand_df):,}")
        st.metric("Positive Ratio", f"{(brand_df['sentiment']=='Positive').mean()*100:.1f}%")
'''

with open("app.py", "a", encoding="utf-8") as f:
    f.write(sentiment_page_code)

print("Sentiment Analysis page added!")

Sentiment Analysis page added!


In [4]:
clusters_page_code = '''
if page == "Brand Clusters":
    st.subheader("Brand Clustering Analysis")
    st.markdown("Brands grouped automatically by customer satisfaction patterns using K-Means clustering")
    
    st.markdown("---")
    
    col1, col2, col3 = st.columns(3)
    for i, cluster_name in enumerate(brand_features['cluster_name'].unique()):
        brands_in_cluster = brand_features[brand_features['cluster_name'] == cluster_name]['brand'].tolist()
        with [col1, col2, col3][i % 3]:
            st.markdown(f"**{cluster_name}**")
            for b in brands_in_cluster:
                st.write(f"- {b}")
    
    st.markdown("---")
    
    fig = px.scatter(brand_features, x='positive_ratio', y='negative_ratio',
                      color='cluster_name', text='brand', size='engagement_score',
                      title="Brand Clusters - Positive vs Negative Ratio",
                      color_discrete_sequence=['#00cc44', '#ff4444', '#ffaa00'])
    fig.update_traces(textposition='top center')
    fig.update_layout(paper_bgcolor="#0e1117", plot_bgcolor="#0e1117", font_color="white")
    st.plotly_chart(fig, use_container_width=True)
    
    st.markdown("### Detailed Brand Metrics")
    st.dataframe(
        brand_features[['brand', 'avg_rating', 'positive_ratio', 'negative_ratio', 
                        'engagement_score', 'cluster_name']].sort_values('avg_rating', ascending=False),
        use_container_width=True
    )
'''

with open("app.py", "a", encoding="utf-8") as f:
    f.write(clusters_page_code)

print("Brand Clusters page added!")

Brand Clusters page added!


In [5]:
crisis_page_code = '''
if page == "Crisis Detection":
    st.subheader("Crisis Detection System")
    st.markdown("Automatically detects unusual spikes in negative sentiment using Isolation Forest")
    
    st.markdown("---")
    
    selected_brand_crisis = st.selectbox("Select a brand to monitor", sorted(df['brand'].unique()), key="crisis_select")
    
    from sklearn.ensemble import IsolationForest
    
    brand_daily = df[df['brand'] == selected_brand_crisis].copy()
    brand_daily['date'] = pd.to_datetime(brand_daily['date'])
    brand_daily['day'] = brand_daily['date'].dt.date
    
    daily_stats = brand_daily.groupby('day').agg(
        total_reviews=('review_id', 'count'),
        avg_rating=('rating', 'mean'),
        negative_count=('sentiment', lambda x: (x == 'Negative').sum()),
    ).reset_index()
    daily_stats['negative_ratio'] = daily_stats['negative_count'] / daily_stats['total_reviews']
    
    if len(daily_stats) >= 30:
        features = daily_stats[['total_reviews', 'avg_rating', 'negative_ratio']]
        iso = IsolationForest(contamination=0.1, random_state=42)
        daily_stats['anomaly'] = iso.fit_predict(features)
        daily_stats['status'] = daily_stats['anomaly'].map({1: 'Normal', -1: 'Crisis Alert'})
        
        latest_status = daily_stats.sort_values('day').iloc[-1]
        
        col1, col2, col3 = st.columns(3)
        col1.metric("Latest Status", latest_status['status'])
        col2.metric("Latest Negative Ratio", f"{latest_status['negative_ratio']*100:.1f}%")
        col3.metric("Total Crisis Days Detected", (daily_stats['status'] == 'Crisis Alert').sum())
        
        normal = daily_stats[daily_stats['status'] == 'Normal']
        crisis = daily_stats[daily_stats['status'] == 'Crisis Alert']
        
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=daily_stats['day'], y=daily_stats['negative_ratio'],
                                  mode='lines', line=dict(color='gray', width=1), showlegend=False))
        fig.add_trace(go.Scatter(x=normal['day'], y=normal['negative_ratio'],
                                  mode='markers', marker=dict(color='#00cc44', size=6), name='Normal Day'))
        fig.add_trace(go.Scatter(x=crisis['day'], y=crisis['negative_ratio'],
                                  mode='markers', marker=dict(color='#ff4444', size=12), name='Crisis Alert'))
        fig.update_layout(title=f"{selected_brand_crisis} - Negative Sentiment Timeline",
                          paper_bgcolor="#0e1117", plot_bgcolor="#0e1117", font_color="white",
                          xaxis_title="Date", yaxis_title="Negative Review Ratio")
        st.plotly_chart(fig, use_container_width=True)
    else:
        st.warning(f"Not enough historical data for {selected_brand_crisis} to run crisis detection (need 30+ days)")
'''

with open("app.py", "a", encoding="utf-8") as f:
    f.write(crisis_page_code)

print("Crisis Detection page added!")

Crisis Detection page added!


In [6]:
import os

files_to_check = [
    "cleaned_reviews.csv",
    "raw_reviews.csv",
    "brand_clusters.csv",
    "sentiment_model.pkl",
    "tfidf_vectorizer.pkl",
    "kmeans_model.pkl",
    "cluster_scaler.pkl"
]

for f in files_to_check:
    if os.path.exists(f):
        size_mb = os.path.getsize(f) / (1024*1024)
        print(f"{f}: {size_mb:.2f} MB")
    else:
        print(f"{f}: NOT FOUND")

cleaned_reviews.csv: 93.37 MB
raw_reviews.csv: 181.68 MB
brand_clusters.csv: 0.00 MB
sentiment_model.pkl: 0.31 MB
tfidf_vectorizer.pkl: 0.37 MB
kmeans_model.pkl: 0.00 MB
cluster_scaler.pkl: 0.00 MB


In [1]:
import pandas as pd
df = pd.read_csv("cleaned_reviews.csv")
print(df.columns.tolist())
print(df.memory_usage(deep=True))

['review_id', 'user_name', 'review_text', 'rating', 'thumbs_up', 'date', 'brand', 'app_version', 'sentiment', 'clean_text']
Index               132
review_id      40393700
user_name      29465299
review_text    58241189
rating          3801760
thumbs_up       3801760
date           32314960
brand          25799133
app_version    25044212
sentiment      27072699
clean_text     47582402
dtype: int64


In [2]:
df_slim = df.drop(columns=['review_text', 'user_name', 'app_version'])
df_slim.to_csv("cleaned_reviews.csv", index=False)

import os
size_mb = os.path.getsize("cleaned_reviews.csv") / (1024*1024)
print(f"New file size: {size_mb:.2f} MB")

New file size: 58.64 MB


In [3]:
import pandas as pd
import os

df = pd.read_csv("cleaned_reviews.csv")
print("Current rows:", len(df))
print("Current size:", os.path.getsize("cleaned_reviews.csv") / (1024*1024), "MB")

# Sample down further - keep 15% of data, stratified by brand and sentiment
df_sample = df.groupby(['brand', 'sentiment'], group_keys=False).apply(
    lambda x: x.sample(frac=0.15, random_state=42)
)

print("\nSampled rows:", len(df_sample))
df_sample.to_csv("cleaned_reviews_app.csv", index=False)

size_mb = os.path.getsize("cleaned_reviews_app.csv") / (1024*1024)
print(f"New file size: {size_mb:.2f} MB")

Current rows: 475220
Current size: 58.639708518981934 MB


C:\Users\bhoom\AppData\Local\Temp\ipykernel_14376\1285966756.py:9: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_sample = df.groupby(['brand', 'sentiment'], group_keys=False).apply(



Sampled rows: 71284
New file size: 8.76 MB


In [4]:
with open("app.py", "r", encoding="utf-8") as f:
    content = f.read()

content = content.replace('pd.read_csv("cleaned_reviews.csv")', 'pd.read_csv("cleaned_reviews_app.csv")')

with open("app.py", "w", encoding="utf-8") as f:
    f.write(content)

print("app.py updated to use cleaned_reviews_app.csv")

# Verify the change
with open("app.py", "r", encoding="utf-8") as f:
    print("cleaned_reviews_app.csv" in f.read())

app.py updated to use cleaned_reviews_app.csv
True


In [5]:
import pandas as pd
df_check = pd.read_csv("cleaned_reviews_app.csv")
print("Rows:", len(df_check))
print("Columns:", df_check.columns.tolist())
print(df_check.head(3))

Rows: 71284
Columns: ['review_id', 'rating', 'thumbs_up', 'date', 'brand', 'sentiment', 'clean_text']
                              review_id  rating  thumbs_up  \
0  cd6ba9b3-9ec1-4594-a4b4-c11f6a6e26e5       1          0   
1  0818045a-90ae-4703-8e04-3a84d3d8ff41       2          0   
2  9bde1b99-bdb0-45d8-a291-989a61269b43       2          0   

                  date    brand sentiment  \
0  2026-06-14 10:12:04  Blinkit  Negative   
1  2026-06-11 09:42:26  Blinkit  Negative   
2  2026-06-03 21:18:56  Blinkit  Negative   

                                          clean_text  
0                         return option its very bad  
1  what is this you cant make cash payment on the...  
2                             fuel saving app for me  


In [6]:
df_sample2 = df_check.groupby(['brand', 'sentiment'], group_keys=False).apply(
    lambda x: x.sample(frac=0.5, random_state=42)
)
df_sample2.to_csv("cleaned_reviews_app.csv", index=False)

import os
print(f"New size: {os.path.getsize('cleaned_reviews_app.csv')/(1024*1024):.2f} MB")

New size: 4.38 MB


C:\Users\bhoom\AppData\Local\Temp\ipykernel_14376\497705441.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_sample2 = df_check.groupby(['brand', 'sentiment'], group_keys=False).apply(


In [1]:
import os

pkl_files = ["sentiment_model.pkl", "tfidf_vectorizer.pkl", "kmeans_model.pkl", "cluster_scaler.pkl"]

for f in pkl_files:
    if os.path.exists(f):
        size = os.path.getsize(f) / 1024  # KB
        print(f"{f}: {size:.2f} KB")
    else:
        print(f"{f}: NOT FOUND")

sentiment_model.pkl: 313.14 KB
tfidf_vectorizer.pkl: 377.59 KB
kmeans_model.pkl: 0.72 KB
cluster_scaler.pkl: 0.68 KB
